[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/AI-in-healthcare-book/blob/main/notebooks/chapter_04/notebook_4_2_missing_data_handling.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 4.2: Missing Data Handling in Healthcare

**Chapter 4: Data Preparation**

## Learning Objectives

By the end of this notebook, you will be able to:
1. Understand different missing data mechanisms (MCAR, MAR, MNAR)
2. Implement various imputation strategies
3. Compare the impact of imputation methods on model performance
4. Visualize missing data patterns
5. Make informed decisions about when to impute vs when to model missingness

## Clinical Context

Missing data in healthcare is **not random**:
- Sicker patients get more tests (MAR)
- Abnormal values trigger follow-up tests (MNAR)
- Equipment failures create random gaps (MCAR)
- Financial barriers prevent testing (MNAR)

**Poor handling of missing data can:**
- Introduce bias
- Reduce statistical power
- Lead to incorrect predictions
- Worsen health disparities

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ Libraries imported")

## 1. Understanding Missing Data Mechanisms

In [ ]:
def generate_complete_dataset(n_patients=1000):
    """Generate complete healthcare dataset (no missing values)."""

    # Demographics
    age = np.random.normal(65, 15, n_patients)
    age = np.clip(age, 18, 100)

    sex = np.random.choice([0, 1], n_patients)  # 0=F, 1=M

    # Risk factors
    bmi = np.random.normal(28, 5, n_patients)
    bmi = np.clip(bmi, 15, 50)

    smoker = np.random.choice([0, 1], n_patients, p=[0.7, 0.3])

    # Labs - correlated with outcome
    glucose = np.random.normal(110, 25, n_patients)
    hba1c = (glucose - 70) / 30 + np.random.normal(0, 0.5, n_patients)
    hba1c = np.clip(hba1c, 4, 14)

    creatinine = np.random.gamma(2, 0.5, n_patients)
    creatinine = np.clip(creatinine, 0.5, 8)

    # Outcome: diabetes risk
    risk_score = (
        0.02 * (age - 65) +
        0.3 * (bmi - 28) / 5 +
        0.5 * smoker +
        0.01 * (glucose - 110) +
        0.2 * (hba1c - 5.5) +
        0.3 * (creatinine - 1.0)
    )

    prob = 1 / (1 + np.exp(-risk_score))
    outcome = (np.random.rand(n_patients) < prob).astype(int)

    df = pd.DataFrame({
        'age': age,
        'sex': sex,
        'bmi': bmi,
        'smoker': smoker,
        'glucose': glucose,
        'hba1c': hba1c,
        'creatinine': creatinine,
        'outcome': outcome
    })

    return df

# Generate complete data
df_complete = generate_complete_dataset(1000)
print(f"Complete dataset: {len(df_complete)} patients")
print(f"Outcome prevalence: {df_complete['outcome'].mean():.1%}")
print(f"\nNo missing values:")
print(df_complete.isnull().sum())
df_complete.head()

### 1.1 MCAR: Missing Completely At Random

In [ ]:
def introduce_mcar(df, columns, missing_rate=0.20):
    """
    MCAR: Missing Completely At Random.

    Missingness is independent of observed and unobserved data.
    Example: Equipment malfunction, random data entry errors.
    """
    df_mcar = df.copy()

    for col in columns:
        missing_mask = np.random.rand(len(df)) < missing_rate
        df_mcar.loc[missing_mask, col] = np.nan

    return df_mcar

# Introduce MCAR missingness
df_mcar = introduce_mcar(df_complete, ['glucose', 'hba1c'], missing_rate=0.20)

print("MCAR Missingness:")
print(df_mcar[['glucose', 'hba1c']].isnull().sum())
print(f"\nMissing rate: {df_mcar['glucose'].isnull().mean():.1%}")

# Check if missingness is related to outcome (it shouldn't be for MCAR)
missing_outcome_rate = df_mcar[df_mcar['glucose'].isnull()]['outcome'].mean()
present_outcome_rate = df_mcar[df_mcar['glucose'].notna()]['outcome'].mean()

print(f"\nOutcome rate (glucose missing): {missing_outcome_rate:.1%}")
print(f"Outcome rate (glucose present): {present_outcome_rate:.1%}")
print(f"Difference: {abs(missing_outcome_rate - present_outcome_rate):.1%} (should be small)")

### 1.2 MAR: Missing At Random

In [ ]:
def introduce_mar(df, target_col, predictor_col, threshold_percentile=70):
    """
    MAR: Missing At Random.

    Missingness depends on observed data, not the missing value itself.
    Example: HbA1c more likely to be ordered when glucose is high.
    """
    df_mar = df.copy()

    # HbA1c more likely missing when glucose is LOW (less clinical concern)
    threshold = df[predictor_col].quantile(threshold_percentile / 100)

    # Higher probability of missing when glucose is below threshold
    prob_missing = np.where(
        df[predictor_col] < threshold,
        0.50,  # 50% missing when glucose low
        0.10   # 10% missing when glucose high
    )

    missing_mask = np.random.rand(len(df)) < prob_missing
    df_mar.loc[missing_mask, target_col] = np.nan

    return df_mar

# Introduce MAR missingness
df_mar = introduce_mar(df_complete, 'hba1c', 'glucose', threshold_percentile=70)

print("MAR Missingness:")
print(f"HbA1c missing: {df_mar['hba1c'].isnull().sum()} ({df_mar['hba1c'].isnull().mean():.1%})")

# Show relationship between glucose and HbA1c missingness
glucose_when_hba1c_missing = df_mar[df_mar['hba1c'].isnull()]['glucose'].mean()
glucose_when_hba1c_present = df_mar[df_mar['hba1c'].notna()]['glucose'].mean()

print(f"\nMean glucose (HbA1c missing): {glucose_when_hba1c_missing:.1f}")
print(f"Mean glucose (HbA1c present): {glucose_when_hba1c_present:.1f}")
print(f"Difference: {abs(glucose_when_hba1c_missing - glucose_when_hba1c_present):.1f} (substantial!)")

### 1.3 MNAR: Missing Not At Random

In [ ]:
def introduce_mnar(df, column, high_values_missing=True):
    """
    MNAR: Missing Not At Random.

    Missingness depends on the unobserved value itself.
    Example: Patients with high creatinine may avoid follow-up tests.
    """
    df_mnar = df.copy()

    if high_values_missing:
        # Higher values more likely to be missing
        percentile_rank = df[column].rank(pct=True)
        prob_missing = percentile_rank  # 0-100% probability
    else:
        # Lower values more likely to be missing
        percentile_rank = df[column].rank(pct=True)
        prob_missing = 1 - percentile_rank

    missing_mask = np.random.rand(len(df)) < prob_missing
    df_mnar.loc[missing_mask, column] = np.nan

    return df_mnar

# Introduce MNAR missingness
df_mnar = introduce_mnar(df_complete, 'creatinine', high_values_missing=True)

print("MNAR Missingness:")
print(f"Creatinine missing: {df_mnar['creatinine'].isnull().sum()} ({df_mnar['creatinine'].isnull().mean():.1%})")

# Compare observed vs true distribution
print(f"\nMean creatinine (observed only): {df_mnar['creatinine'].mean():.2f}")
print(f"Mean creatinine (true complete): {df_complete['creatinine'].mean():.2f}")
print(f"Bias: {df_complete['creatinine'].mean() - df_mnar['creatinine'].mean():.2f}")
print("\n⚠️ Observed values are biased LOW (high values missing)!")

## 2. Visualize Missing Data Patterns

In [ ]:
# Create dataset with mixed missingness mechanisms
df_mixed = df_complete.copy()
df_mixed.loc[np.random.rand(len(df_mixed)) < 0.15, 'glucose'] = np.nan  # MCAR
df_mixed = introduce_mar(df_mixed, 'hba1c', 'glucose', threshold_percentile=70)  # MAR
df_mixed = introduce_mnar(df_mixed, 'creatinine', high_values_missing=True)  # MNAR

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Missing data matrix
ax = axes[0, 0]
missing_matrix = df_mixed[['age', 'bmi', 'glucose', 'hba1c', 'creatinine']].isnull()
sns.heatmap(missing_matrix.iloc[:100], cbar=False, yticklabels=False, ax=ax, cmap='RdYlGn_r')
ax.set_title('Missing Data Pattern (First 100 patients)', fontweight='bold')

# Missing percentage by column
ax = axes[0, 1]
missing_pct = (df_mixed.isnull().sum() / len(df_mixed)) * 100
missing_pct = missing_pct[missing_pct > 0]
missing_pct.plot(kind='barh', ax=ax, color='coral')
ax.set_xlabel('Missing %')
ax.set_title('Missing Data by Variable', fontweight='bold')
ax.grid(True, alpha=0.3)

# Glucose distribution: missing vs present HbA1c (MAR pattern)
ax = axes[1, 0]
df_mixed[df_mixed['hba1c'].isnull()]['glucose'].hist(bins=30, alpha=0.6, label='HbA1c Missing', ax=ax, color='red')
df_mixed[df_mixed['hba1c'].notna()]['glucose'].hist(bins=30, alpha=0.6, label='HbA1c Present', ax=ax, color='blue')
ax.set_xlabel('Glucose (mg/dL)')
ax.set_ylabel('Count')
ax.set_title('MAR Pattern: Glucose vs HbA1c Missingness', fontweight='bold')
ax.legend()

# Creatinine: observed vs true (MNAR pattern)
ax = axes[1, 1]
ax.hist(df_complete['creatinine'], bins=30, alpha=0.6, label='True Complete', color='green')
ax.hist(df_mixed['creatinine'].dropna(), bins=30, alpha=0.6, label='Observed Only', color='orange')
ax.axvline(df_complete['creatinine'].mean(), color='green', linestyle='--', linewidth=2, label='True Mean')
ax.axvline(df_mixed['creatinine'].mean(), color='orange', linestyle='--', linewidth=2, label='Observed Mean')
ax.set_xlabel('Creatinine (mg/dL)')
ax.set_ylabel('Count')
ax.set_title('MNAR Pattern: Creatinine Distribution Bias', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.show()

print("\n📊 Key Observations:")
print("- MCAR (glucose): Random pattern, no bias")
print("- MAR (HbA1c): Missing when glucose low (observable pattern)")
print("- MNAR (creatinine): High values missing (unobservable, creates bias)")

## 3. Imputation Strategies

### 3.1 Simple Imputation

In [ ]:
def simple_imputation(df, strategy='mean'):
    """
    Simple imputation: mean, median, or most frequent.

    Pros: Fast, deterministic
    Cons: Doesn't preserve relationships, reduces variance
    """
    df_imputed = df.copy()

    numeric_cols = df.select_dtypes(include=[np.number]).columns
    numeric_cols = [col for col in numeric_cols if col != 'outcome']

    imputer = SimpleImputer(strategy=strategy)
    df_imputed[numeric_cols] = imputer.fit_transform(df[numeric_cols])

    return df_imputed, imputer

# Test different simple strategies
df_mean, _ = simple_imputation(df_mixed, strategy='mean')
df_median, _ = simple_imputation(df_mixed, strategy='median')

print("Simple Imputation Results:")
print(f"\nOriginal glucose (with missing): mean={df_mixed['glucose'].mean():.1f}, median={df_mixed['glucose'].median():.1f}")
print(f"After mean imputation: mean={df_mean['glucose'].mean():.1f}, std={df_mean['glucose'].std():.1f}")
print(f"After median imputation: mean={df_median['glucose'].mean():.1f}, std={df_median['glucose'].std():.1f}")
print(f"True complete data: mean={df_complete['glucose'].mean():.1f}, std={df_complete['glucose'].std():.1f}")
print("\n⚠️ Note: Variance is artificially reduced after simple imputation")

### 3.2 KNN Imputation

In [ ]:
def knn_imputation(df, n_neighbors=5):
    """
    KNN imputation: Use k nearest neighbors to impute.

    Pros: Preserves relationships between features
    Cons: Slower, sensitive to scaling
    """
    df_imputed = df.copy()

    numeric_cols = df.select_dtypes(include=[np.number]).columns
    numeric_cols = [col for col in numeric_cols if col != 'outcome']

    imputer = KNNImputer(n_neighbors=n_neighbors)
    df_imputed[numeric_cols] = imputer.fit_transform(df[numeric_cols])

    return df_imputed, imputer

df_knn, _ = knn_imputation(df_mixed, n_neighbors=5)

print("KNN Imputation Results:")
print(f"After KNN imputation: glucose mean={df_knn['glucose'].mean():.1f}, std={df_knn['glucose'].std():.1f}")
print(f"True complete data: glucose mean={df_complete['glucose'].mean():.1f}, std={df_complete['glucose'].std():.1f}")
print("\n✓ KNN better preserves variance than simple imputation")

### 3.3 Iterative Imputation (MICE)

In [ ]:
def mice_imputation(df, max_iter=10):
    """
    MICE: Multivariate Imputation by Chained Equations.

    Iteratively models each feature with missing values as a function of other features.

    Pros: Sophisticated, preserves relationships
    Cons: Slow, requires sufficient data
    """
    df_imputed = df.copy()

    numeric_cols = df.select_dtypes(include=[np.number]).columns
    numeric_cols = [col for col in numeric_cols if col != 'outcome']

    imputer = IterativeImputer(max_iter=max_iter, random_state=42)
    df_imputed[numeric_cols] = imputer.fit_transform(df[numeric_cols])

    return df_imputed, imputer

df_mice, _ = mice_imputation(df_mixed, max_iter=10)

print("MICE Imputation Results:")
print(f"After MICE: glucose mean={df_mice['glucose'].mean():.1f}, std={df_mice['glucose'].std():.1f}")
print(f"True complete data: glucose mean={df_complete['glucose'].mean():.1f}, std={df_complete['glucose'].std():.1f}")
print("\n✓ MICE often best preserves complex relationships")

### 3.4 Modeling Missingness as a Feature

In [ ]:
def impute_with_missingness_indicator(df):
    """
    Impute AND create indicator variables for missingness.

    Useful when missingness itself is informative (MAR/MNAR).
    """
    df_imputed = df.copy()

    numeric_cols = df.select_dtypes(include=[np.number]).columns
    numeric_cols = [col for col in numeric_cols if col != 'outcome']

    # Create missingness indicators
    for col in numeric_cols:
        if df[col].isnull().any():
            df_imputed[f'{col}_missing'] = df[col].isnull().astype(int)

    # Simple imputation
    imputer = SimpleImputer(strategy='median')
    df_imputed[numeric_cols] = imputer.fit_transform(df[numeric_cols])

    return df_imputed

df_indicator = impute_with_missingness_indicator(df_mixed)

print("Imputation with Missingness Indicators:")
print(f"\nNew indicator columns created:")
indicator_cols = [col for col in df_indicator.columns if '_missing' in col]
print(indicator_cols)

print(f"\nMissingness rate by outcome:")
print(df_indicator.groupby('outcome')[indicator_cols].mean())
print("\n💡 If rates differ by outcome, missingness is informative!")

## 4. Compare Impact on Model Performance

In [ ]:
def evaluate_imputation_method(df, method_name):
    """
    Train model and evaluate on test set.
    """
    # Prepare features
    feature_cols = [col for col in df.columns if col not in ['outcome']]
    X = df[feature_cols]
    y = df['outcome']

    # Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    # Train
    clf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
    clf.fit(X_train, y_train)

    # Evaluate
    y_pred = clf.predict(X_test)
    y_pred_proba = clf.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)

    return {'method': method_name, 'accuracy': acc, 'auroc': auc}

# Evaluate all methods
results = []

# Ground truth (complete data)
results.append(evaluate_imputation_method(df_complete, 'Ground Truth (No Missing)'))

# Simple imputation
df_mean, _ = simple_imputation(df_mixed, strategy='mean')
results.append(evaluate_imputation_method(df_mean, 'Simple (Mean)'))

df_median, _ = simple_imputation(df_mixed, strategy='median')
results.append(evaluate_imputation_method(df_median, 'Simple (Median)'))

# KNN
df_knn, _ = knn_imputation(df_mixed, n_neighbors=5)
results.append(evaluate_imputation_method(df_knn, 'KNN (k=5)'))

# MICE
df_mice, _ = mice_imputation(df_mixed, max_iter=10)
results.append(evaluate_imputation_method(df_mice, 'MICE'))

# With indicators
df_indicator = impute_with_missingness_indicator(df_mixed)
results.append(evaluate_imputation_method(df_indicator, 'Median + Indicators'))

# Display results
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('auroc', ascending=False)

print("="*70)
print("IMPUTATION METHOD COMPARISON")
print("="*70)
print(results_df.to_string(index=False))
print("="*70)

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# AUROC comparison
ax = axes[0]
results_df_sorted = results_df.sort_values('auroc')
colors = ['green' if x == 'Ground Truth (No Missing)' else 'steelblue' for x in results_df_sorted['method']]
ax.barh(results_df_sorted['method'], results_df_sorted['auroc'], color=colors)
ax.set_xlabel('AUROC')
ax.set_title('Model Performance by Imputation Method', fontweight='bold')
ax.axvline(results_df[results_df['method'] == 'Ground Truth (No Missing)']['auroc'].values[0],
           color='green', linestyle='--', linewidth=2, alpha=0.5, label='Ground Truth')
ax.grid(True, alpha=0.3, axis='x')
ax.legend()

# Performance gap from ground truth
ax = axes[1]
ground_truth_auroc = results_df[results_df['method'] == 'Ground Truth (No Missing)']['auroc'].values[0]
results_df_sorted = results_df[results_df['method'] != 'Ground Truth (No Missing)'].copy()
results_df_sorted['gap'] = ground_truth_auroc - results_df_sorted['auroc']
results_df_sorted = results_df_sorted.sort_values('gap')

colors = ['red' if x > 0.02 else 'orange' if x > 0.01 else 'lightgreen' for x in results_df_sorted['gap']]
ax.barh(results_df_sorted['method'], results_df_sorted['gap'], color=colors)
ax.set_xlabel('AUROC Gap from Ground Truth')
ax.set_title('Performance Loss Due to Missing Data', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 5. Time Series: Forward/Backward Fill

In [ ]:
def generate_time_series_with_gaps(n_patients=10, hours=48):
    """
    Generate time series vital signs with random gaps.
    """
    data = []

    for patient_id in range(n_patients):
        for hour in range(hours):
            # Random gaps (30% missing)
            if np.random.rand() < 0.30:
                hr = np.nan
                sbp = np.nan
            else:
                hr = 70 + np.random.normal(0, 10) + hour * 0.2  # Slight upward trend
                sbp = 120 + np.random.normal(0, 15) + hour * 0.1

            data.append({
                'patient_id': patient_id,
                'hour': hour,
                'heart_rate': hr,
                'sbp': sbp
            })

    return pd.DataFrame(data)

df_ts = generate_time_series_with_gaps(n_patients=5, hours=48)

print("Time Series Data with Gaps:")
print(f"Total records: {len(df_ts)}")
print(f"Heart rate missing: {df_ts['heart_rate'].isnull().sum()} ({df_ts['heart_rate'].isnull().mean():.1%})")
print("\nFirst patient sample:")
print(df_ts[df_ts['patient_id'] == 0].head(15))

In [ ]:
# Apply forward fill and backward fill
df_ts_ffill = df_ts.copy()
df_ts_ffill['heart_rate'] = df_ts_ffill.groupby('patient_id')['heart_rate'].ffill()
df_ts_ffill['sbp'] = df_ts_ffill.groupby('patient_id')['sbp'].ffill()

df_ts_bfill = df_ts.copy()
df_ts_bfill['heart_rate'] = df_ts_bfill.groupby('patient_id')['heart_rate'].bfill()
df_ts_bfill['sbp'] = df_ts_bfill.groupby('patient_id')['sbp'].bfill()

# Combined: forward fill then backward fill
df_ts_combined = df_ts.copy()
df_ts_combined['heart_rate'] = df_ts_combined.groupby('patient_id')['heart_rate'].ffill().bfill()
df_ts_combined['sbp'] = df_ts_combined.groupby('patient_id')['sbp'].ffill().bfill()

# Visualize
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

patient = 0
subset_original = df_ts[df_ts['patient_id'] == patient]
subset_ffill = df_ts_ffill[df_ts_ffill['patient_id'] == patient]
subset_combined = df_ts_combined[df_ts_combined['patient_id'] == patient]

# Original with gaps
ax = axes[0]
ax.plot(subset_original['hour'], subset_original['heart_rate'], 'o-', label='Observed', markersize=4)
ax.set_ylabel('Heart Rate (bpm)')
ax.set_title(f'Original Data with Missing Values (Patient {patient})', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Forward fill
ax = axes[1]
ax.plot(subset_original['hour'], subset_original['heart_rate'], 'o', label='Observed', markersize=4, alpha=0.5)
ax.plot(subset_ffill['hour'], subset_ffill['heart_rate'], '-', label='Forward Fill', linewidth=2)
ax.set_ylabel('Heart Rate (bpm)')
ax.set_title('Forward Fill (ffill)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Combined
ax = axes[2]
ax.plot(subset_original['hour'], subset_original['heart_rate'], 'o', label='Observed', markersize=4, alpha=0.5)
ax.plot(subset_combined['hour'], subset_combined['heart_rate'], '-', label='Forward + Backward Fill', linewidth=2, color='green')
ax.set_xlabel('Hour')
ax.set_ylabel('Heart Rate (bpm)')
ax.set_title('Combined Fill (ffill + bfill)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Time Series Imputation Notes:")
print("- Forward fill: Assume value persists (good for slow-changing vitals)")
print("- Backward fill: Use future value (breaks causality, use with caution!)")
print("- Limit forward fill duration to avoid stale values (e.g., max 4 hours)")

## 6. When NOT to Impute

In [ ]:
print("="*70)
print("WHEN NOT TO IMPUTE")
print("="*70)
print("\n1. HIGH MISSINGNESS (>50%)")
print("   - Imputation unreliable, consider dropping variable")
print("   - Or collect more data before modeling")

print("\n2. MNAR WITH CLINICAL MEANING")
print("   - Test not ordered → patient not sick enough")
print("   - Test not ordered → patient too sick (comfort care)")
print("   - Missing IS the signal → use indicator variable")

print("\n3. OUTCOME VARIABLE")
print("   - NEVER impute outcome")
print("   - Exclude these samples or collect labels")

print("\n4. IDENTIFIERS AND PROTECTED ATTRIBUTES")
print("   - Don't impute patient ID, race, etc.")
print("   - Missing may indicate data quality issue")

print("\n5. WHEN MISSINGNESS PATTERN IS THE FEATURE")
print("   - Number of missed appointments")
print("   - Engagement with healthcare system")
print("   - Use counts/patterns directly")
print("="*70)

# Example: High missingness
df_high_missing = df_complete.copy()
df_high_missing.loc[np.random.rand(len(df_high_missing)) < 0.70, 'creatinine'] = np.nan

print(f"\nExample: Creatinine with 70% missing")
print(f"Missing: {df_high_missing['creatinine'].isnull().sum()} / {len(df_high_missing)} ({df_high_missing['creatinine'].isnull().mean():.1%})")
print(f"Observed mean: {df_high_missing['creatinine'].mean():.2f}")
print(f"True mean: {df_complete['creatinine'].mean():.2f}")
print(f"Standard error of mean: {df_high_missing['creatinine'].std() / np.sqrt(df_high_missing['creatinine'].notna().sum()):.3f}")
print("\n❌ Recommendation: DROP this variable or collect more data")

## 7. Practical Guidelines

In [ ]:
def missing_data_workflow(df, outcome_col='outcome'):
    """
    Systematic workflow for handling missing data.
    """
    print("="*70)
    print("MISSING DATA WORKFLOW")
    print("="*70)

    # Step 1: Quantify missingness
    print("\nStep 1: QUANTIFY MISSINGNESS")
    print("-" * 70)
    missing_summary = pd.DataFrame({
        'Missing_Count': df.isnull().sum(),
        'Missing_Pct': (df.isnull().sum() / len(df)) * 100
    }).sort_values('Missing_Pct', ascending=False)

    print(missing_summary[missing_summary['Missing_Count'] > 0])

    # Step 2: Assess mechanism
    print("\nStep 2: ASSESS MECHANISM (MCAR/MAR/MNAR)")
    print("-" * 70)
    print("For each variable with missing values:")

    for col in df.columns:
        if df[col].isnull().any() and col != outcome_col:
            # Check if missingness relates to outcome
            missing_mask = df[col].isnull()
            if outcome_col in df.columns:
                outcome_rate_missing = df[missing_mask][outcome_col].mean()
                outcome_rate_present = df[~missing_mask][outcome_col].mean()
                diff = abs(outcome_rate_missing - outcome_rate_present)

                if diff > 0.05:
                    print(f"  {col}: Likely MAR/MNAR (outcome rate differs by {diff:.1%})")
                else:
                    print(f"  {col}: Likely MCAR (outcome rate similar)")

    # Step 3: Recommend strategy
    print("\nStep 3: RECOMMENDED STRATEGY")
    print("-" * 70)

    for col in df.columns:
        if df[col].isnull().any() and col != outcome_col:
            missing_pct = (df[col].isnull().sum() / len(df)) * 100

            if missing_pct > 50:
                print(f"  {col} ({missing_pct:.0f}% missing): DROP or collect more data")
            elif missing_pct > 20:
                print(f"  {col} ({missing_pct:.0f}% missing): MICE or KNN + missingness indicator")
            else:
                print(f"  {col} ({missing_pct:.0f}% missing): Simple imputation acceptable")

    print("\n" + "="*70)

# Run workflow
missing_data_workflow(df_mixed, outcome_col='outcome')

## 8. Key Takeaways

### What We Learned

1. **Missing Data Mechanisms**:
   - **MCAR**: Rare in healthcare, truly random
   - **MAR**: Common, depends on observed data
   - **MNAR**: Most problematic, depends on unobserved data

2. **Imputation Strategies**:
   - **Simple**: Fast, but reduces variance
   - **KNN**: Preserves relationships between features
   - **MICE**: Sophisticated, best for complex data
   - **Indicators**: Capture information in missingness itself

3. **Performance Impact**:
   - Poor imputation degrades model performance
   - Missingness indicators can improve predictions
   - No single best method - depends on mechanism

4. **Time Series**:
   - Forward fill assumes persistence
   - Backward fill breaks causality
   - Limit fill duration to avoid stale values

5. **When NOT to Impute**:
   - High missingness (>50%)
   - MNAR with clinical meaning
   - Outcome variable
   - When pattern is the feature

### Clinical Considerations

- **Lab not ordered ≠ lab normal**: Missingness often informative
- **Socioeconomic factors**: Financial barriers create MNAR patterns
- **Documentation burden**: Providers skip non-critical fields
- **Clinical judgment**: Understand WHY data is missing
- **Bias amplification**: Poor imputation worsens health disparities

### Best Practices

1. **Always visualize** missing data patterns first
2. **Assess mechanism** before choosing method
3. **Compare multiple strategies** on held-out validation set
4. **Document assumptions** about missingness mechanism
5. **Consider indicators** for variables with >10% missing
6. **Validate with domain experts** to understand clinical context
7. **Report sensitivity** to imputation method

---

*This notebook is part of "AI in Healthcare" (Volume 1, Chapter 4: Data Preparation)*

## Exercises

1. **Mechanism Detection**: Create a dataset where HbA1c is MNAR (high values more likely missing). Compare observed vs true distributions.

2. **Impact Analysis**: Generate data with 40% MNAR missingness. Compare model performance with and without missingness indicators.

3. **Time Series Challenge**: Implement a "smart" forward fill that limits fill duration to 4 hours. Compare to unlimited forward fill.

4. **Cross-Validation**: Implement proper cross-validation where imputation is done within each fold (not on the full dataset).

5. **Real-World Scenario**: Simulate a multi-site study where missingness patterns differ by hospital. Explore how this affects model generalization.